[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/MCHP-ELECTRE-TRI-B/blob/main/example.ipynb)

# MCHP ELECTRE TRI-B — a small illustrative example

This notebook shows how to use the implementation in [`methods.py`](methods.py) on a **small fictitious problem**.

We sort five imaginary countries into three ordered categories (**Low**, **Medium**, **High**) using a *hierarchy* of criteria. 

- All criteria are **to be maximized** (higher value = better performance). 
- A cost criterion should be converted beforehand, e.g. by taking its negative.


Reference for MCHP on ELECTRE-TRI methods:

 - Corrente, S., Greco, S., & Słowiński, R. (2016). Multiple criteria hierarchy process for ELECTRE Tri methods. *European Journal of Operational Research*, 252(1), 191–203.

## 0. Setup

In [10]:
from pathlib import Path
import urllib.request

if not Path("methods.py").exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/diogoflim/MCHP-ELECTRE-TRI-B/main/methods.py",
        "methods.py",
    )


In [11]:
import pandas as pd

from methods import (
    CriterionNode,
    HierarchicalElectreTriB,
    set_equal_weights_topdown,
    compute_lambdas,
    compute_hierarchy,
)

## 1. The hierarchy of criteria

Internal nodes carry a `name`; leaves additionally carry a `leaf_id`, which is the column name of the
performance table.

```
Overall Quality of Life
├── Freedom
│   ├── Economic Freedom   (econ)
│   └── Political Freedom  (pol)
└── Prosperity
    ├── Income             (inc)
    └── Health             (hea)
```

In [12]:
root = CriterionNode("Overall Quality of Life", [
    CriterionNode("Freedom", [
        CriterionNode("Economic Freedom", leaf_id="econ"),
        CriterionNode("Political Freedom", leaf_id="pol"),
    ]),
    CriterionNode("Prosperity", [
        CriterionNode("Income", leaf_id="inc"),
        CriterionNode("Health", leaf_id="hea"),
    ]),
])

leaves = root.leaves()
leaves

['econ', 'pol', 'inc', 'hea']

## 2. Performance table

Five fictitious countries evaluated on the four elementary criteria (scale 0–100).

In [13]:
df = pd.DataFrame(
    {
        "econ": [85, 75, 30, 25, 95],
        "pol":  [80, 72, 35, 20, 92],
        "inc":  [90, 45, 80, 35, 88],
        "hea":  [78, 50, 75, 30, 25],
    },
    index=["Alfaria", "Belvia", "Cordoria", "Dunmara", "Elbonia"],
)
df

,econ,pol,inc,hea
Alfaria,85,80,90,78
Belvia,75,72,45,50
Cordoria,30,35,80,75
Dunmara,25,20,35,30
Elbonia,95,92,88,25


## 3. Categories, boundary profiles and preference parameters

Three ordered categories require two boundary profiles, `b1` (Low / Medium) and `b2` (Medium / High).

For each elementary criterion we set an indifference threshold `q`, a preference threshold `p` and a
veto threshold `v`.

Weights are split equally top-down (each of the two macro-criteria receives 0.5, each leaf 0.25), and the majority thresholds `lambda` are set to 65% of the weight sum below each node, so that the same majority requirement applies at every level of the hierarchy.

In [14]:
categories = ["Low", "Medium", "High"]

profiles = {
    "b1": {c: 40.0 for c in leaves},   # Low  | Medium
    "b2": {c: 70.0 for c in leaves},   # Medium | High
}

q = {c:  5.0 for c in leaves}   # indifference
p = {c: 15.0 for c in leaves}   # preference
v = {c: 40.0 for c in leaves}   # veto

weights = set_equal_weights_topdown(root, 1.0)
lambdas = compute_lambdas(root, weights, 0.65)

print("weights:", weights)
print("lambdas:", lambdas)

weights: {'econ': 0.25, 'pol': 0.25, 'inc': 0.25, 'hea': 0.25}
lambdas: {'Overall Quality of Life': 0.65, 'Freedom': 0.325, 'Prosperity': 0.325}


## 4. Building the model and sorting the alternatives

`relation` selects the outranking relation used at every node:

| value | meaning |
|-------|---------|
| `O1`  | concordance only (no veto) |
| `O2`  | concordance + veto (default) |
| `O3`  | credibility index compared to `lambda` |

`assign_all_nodes` returns the category of the alternative for the root **and** for every internal node.

In [15]:
model = HierarchicalElectreTriB(
    root=root,
    weights=weights,
    q=q, p=p, v=v,
    lambdas=lambdas,
    profiles=profiles,
    categories=categories,
    relation="O2",
)

def sort_all(model, procedure="pessimistic"):
    """Assignments of every alternative at every node of the hierarchy."""
    return pd.DataFrame(
        {a: model.assign_all_nodes(df.loc[a], procedure) for a in df.index}
    ).T

results = sort_all(model, "pessimistic")
results

,Overall Quality of Life,Freedom,Prosperity
Alfaria,High,High,High
Belvia,Medium,High,Medium
Cordoria,Medium,Medium,High
Dunmara,Low,Low,Medium
Elbonia,Medium,High,Low


Reading the table: **Elbonia** is excellent on both freedom criteria and on income, but its very poor health performance keeps *Prosperity* in the **Low** category. It is an insight that a non-hierarchical sorting at the root only would have hidden.

In [16]:
compute_hierarchy("Elbonia", root, df, results);

Country: Elbonia
Overall Quality of Life                                 -> Medium
  Freedom                                               -> High
    Economic Freedom                                       (value = 95.00)
    Political Freedom                                      (value = 92.00)
  Prosperity                                            -> Low
    Income                                                 (value = 88.00)
    Health                                                 (value = 25.00)


## 5. Comparing the assignment procedures

- **pessimistic**: the alternative must outrank the profile;
- **optimistic**: the profile must be preferred to the alternative;
- **optimistic_modified**: the optimistic rule corrected so that a node is never assigned to a category
  lower than the worst category of its sub-criteria.

In [17]:
pd.concat(
    {proc: sort_all(model, proc)
     for proc in ["pessimistic", "optimistic", "optimistic_modified"]},
    axis=1,
)

pessimistic                                 optimistic  \
         Overall Quality of Life Freedom Prosperity Overall Quality of Life   
Alfaria                     High    High       High                    High   
Belvia                    Medium    High     Medium                  Medium   
Cordoria                  Medium  Medium       High                  Medium   
Dunmara                      Low     Low     Medium                     Low   
Elbonia                   Medium    High        Low                    High   

                                optimistic_modified                     
         Freedom Prosperity Overall Quality of Life Freedom Prosperity  
Alfaria     High       High                    High    High       High  
Belvia      High     Medium                  Medium    High     Medium  
Cordoria  Medium       High                  Medium  Medium       High  
Dunmara      Low     Medium                     Low     Low     Medium  
Elbonia     High       High                    High    High       High

## 6. Effect of the veto

Compare `O1` (no veto) with `O2` (veto). Under `O1`, Elbonia's terrible health score is compensated by
its other performances and the country reaches **High** overall; under `O2` the veto blocks that
outranking and Elbonia drops to **Medium**.

In [18]:
comparison = pd.DataFrame({
    rel: sort_all(
        HierarchicalElectreTriB(root, weights, q, p, v, lambdas, profiles, categories, rel),
        "pessimistic",
    )["Overall Quality of Life"]
    for rel in ["O1", "O2", "O3"]
})
comparison

,O1,O2,O3
Alfaria,High,High,High
Belvia,Medium,Medium,Medium
Cordoria,Medium,Medium,Medium
Dunmara,Low,Low,Low
Elbonia,High,Medium,Medium
